In [ ]:
# @title Install only the modified local checkout (run this cell yourself).

from pathlib import Path
import importlib
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/disco_rl')]
repo_root = next((p.resolve() for p in candidates
                  if (p / 'disco_rl' / 'continuous_policies.py').is_file()
                  and ((p / 'pyproject.toml').is_file()
                       or (p / 'setup.py').is_file())), None)
if repo_root is None:
  raise RuntimeError(
      'Modified checkout unavailable. Upload/clone the continuous-action '
      'repository to /content/disco_rl, or run from its root or colabs '
      'directory. Do not install the unchanged upstream repository.')
if any(name == 'disco_rl' or name.startswith('disco_rl.')
       for name in sys.modules):
  raise RuntimeError('Restart the kernel, then run this local-install cell first.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(repo_root)])
sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()
import disco_rl
if Path(disco_rl.__file__).resolve().parent != repo_root / 'disco_rl':
  raise RuntimeError('disco_rl did not resolve to the modified checkout; restart the kernel.')
print(f'Using modified source: {disco_rl.__file__}')

from typing import Any
from functools import partial

import chex
import distrax
import jax
import jax.numpy as jnp
from matplotlib import pyplot as plt
from ml_collections import config_dict
import numpy as np
import optax
import pandas as pd
import seaborn as sns
import tqdm

# Types & utils
from disco_rl import types
from disco_rl import utils
from disco_rl import continuous_policies

# Environments
from disco_rl.environments import base as base_env
from disco_rl.environments import jittable_envs

# Learning
from disco_rl import agent as agent_lib
from disco_rl.value_fns import value_fn

axis_name = 'i'  # for parallelisation

In [ ]:
# @title Continuous-policy configuration (no pretrained checkpoint).

num_samples = 48
kde_bandwidth = 0.1
wasserstein_smoothing = 0.05
action_minimum = -1.0
action_maximum = 1.0
kde_kwargs = dict(bandwidth=kde_bandwidth,
                  minimum=action_minimum, maximum=action_maximum)
print(dict(num_samples=num_samples, **kde_kwargs,
           wasserstein_smoothing=wasserstein_smoothing))

In [ ]:
# @title Configure a simple MLP agent.


def get_env(batch_size: int) -> base_env.Environment:
  return jittable_envs.ContinuousCatchJittableEnvironment(
      batch_size=batch_size,
      env_settings=jittable_envs.get_config_continuous_catch(),
  )


# Create a dummy environment.
env = get_env(batch_size=1)

# Create settings for an agent.
agent_settings = agent_lib.get_settings_disco(
    continuous_actions=True, num_samples=num_samples,
    kde_bandwidth=kde_bandwidth,
    wasserstein_smoothing=wasserstein_smoothing,
    action_minimum=action_minimum, action_maximum=action_maximum,
)
agent_settings.net_settings.name = 'mlp'
agent_settings.net_settings.net_args = dict(
    dense=(32, 32),
    model_arch_name='lstm',
    head_w_init_std=1e-4,
    model_kwargs=dict(
        head_mlp_hiddens=(32,),
        lstm_size=32,
    ),
)
agent_settings.learning_rate = 5e-4
agent_settings.update_rule.net.hidden_size = 32
agent_settings.update_rule.net.meta_rnn_kwargs.hidden_size = 32

# Create the agent.
agent = agent_lib.Agent(
    agent_settings=agent_settings,
    single_observation_spec=env.single_observation_spec(),
    single_action_spec=env.single_action_spec(),
    batch_axis_name=axis_name,
)

# This architecture requires newly initialized meta-parameters.
random_update_rule_params, _ = agent.update_rule.init_params(
    jax.random.PRNGKey(1)
)
print('Random continuous-action update rule initialized.')
print(dict(continuous_actions=agent_settings.update_rule.continuous_actions,
           num_samples=agent_settings.update_rule.num_samples,
           kde_bandwidth=agent_settings.update_rule.kde_bandwidth,
           wasserstein_smoothing=agent_settings.update_rule.wasserstein_smoothing))

In [ ]:
# @title Shared helper for interacting with jittable environments.
from disco_rl.environments.rollouts import unroll_jittable_actor


In [ ]:
# @title Meta training agent


@chex.dataclass
class MetaTrainState:
  learner_state: agent_lib.LearnerState
  actor_state: types.HaikuState
  value_state: types.ValueState
  env_state: Any
  env_timestep: types.EnvironmentTimestep


class MetaTrainAgent:
  """Meta training agent bundles together all meta training-related funct-ty."""

  agent: agent_lib.Agent
  value_fn: value_fn.ValueFunction
  env: base_env.Environment

  def __init__(
      self,
      batch_size_per_device: int,
      agent_settings: config_dict.ConfigDict,
      value_fn_config: types.ValueFnConfig,
      axis_name: str | None = axis_name,
  ):
    self.env = get_env(batch_size_per_device)
    self.batch_size_per_device = batch_size_per_device
    self.agent = agent_lib.Agent(
        agent_settings=agent_settings,
        single_observation_spec=self.env.single_observation_spec(),
        single_action_spec=self.env.single_action_spec(),
        batch_axis_name=axis_name,
    )
    self.value_fn = value_fn.ValueFunction(value_fn_config, axis_name=axis_name)
    self._unroll_jittable_actor = jax.jit(
        unroll_jittable_actor,
        static_argnames=('env', 'rollout_len', 'actor_step_fn'),
    )

  @property
  def learner_step(self):
    return self.agent.learner_step

  @property
  def actor_step(self):
    return self.agent.actor_step

  @property
  def unroll_net(self):
    return self.agent.unroll_net

  def init_state(self, rng_key: chex.PRNGKey) -> MetaTrainState:
    """Initialize the meta train state."""
    dummy_obs = utils.zeros_like_spec(
        self.env.single_observation_spec(),
        prepend_shape=(self.batch_size_per_device,),
    )
    rng_keys = jax.random.split(rng_key, 3)
    env_state, env_timestep = self.env.reset(rng_key)
    return MetaTrainState(
        learner_state=self.agent.initial_learner_state(rng_keys[0]),
        actor_state=self.agent.initial_actor_state(rng_keys[1]),
        value_state=self.value_fn.initial_state(rng_keys[2], dummy_obs),
        env_state=env_state,
        env_timestep=env_timestep,
    )

  def unroll_actor(
      self, state: MetaTrainState, rng: chex.PRNGKey, rollout_len: int
  ) -> tuple[MetaTrainState, types.ActorRollout]:
    """Unrolls the actor for the given number of steps and updates the state."""
    rollout, new_actor_state, new_env_timestep, new_env_state = (
        self._unroll_jittable_actor(
            state.learner_state.params,
            state.actor_state,
            state.env_timestep,
            state.env_state,
            rng,
            self.env,
            rollout_len,
            self.actor_step,
        )
    )
    new_state = MetaTrainState(
        learner_state=state.learner_state,
        actor_state=new_actor_state,
        value_state=state.value_state,
        env_state=new_env_state,
        env_timestep=new_env_timestep,
    )
    return new_state, rollout


In [ ]:
# @title Calculate meta gradients for a single agent.


def calculate_meta_gradient(
    update_rule_params: types.MetaParams,
    agent_state: MetaTrainState,
    train_rollouts: types.ActorRollout,
    valid_rollout: types.ActorRollout,
    rng: chex.PRNGKey,
    agent: MetaTrainAgent,
    axis_name: str | None = axis_name,
    regularizer_scale: float = 1.0,
):
  """Calculates meta gradients for a single agent."""

  unroll_len = train_rollouts.rewards.shape[0]

  def _inner_step(carry, inputs):
    """Updates agent's and value fn's params using the current update rule."""
    update_rule_params, learner_state, actor_state, value_state = carry
    actor_rollout, learner_rng = inputs

    # Update agent's parameters.
    new_learner_state, new_actor_state, metrics = agent.learner_step(
        rng=learner_rng,
        rollout=actor_rollout,
        learner_state=learner_state,
        agent_net_state=actor_state,
        update_rule_params=update_rule_params,
        is_meta_training=True,
    )

    # Update value function.
    agent_out, _ = agent.unroll_net(
        learner_state.params, actor_state, actor_rollout
    )
    new_value_state, _, _ = agent.value_fn.update(
        value_state, actor_rollout, agent_out['logits']
    )

    return (
        update_rule_params,
        new_learner_state,
        new_actor_state,
        new_value_state,
    ), metrics

  def _outer_loss(
      update_rule_params: types.MetaParams,
      agent_state: MetaTrainState,
      train_rollouts: types.ActorRollout,
      valid_rollout: types.ActorRollout,
      rng: chex.PRNGKey,
  ):
    """Calculates (meta) loss for the update rule."""
    train_rng = rng

    # Perform N inner steps (i.e. agent's params' updates).
    learner_rngs = jax.random.split(train_rng, unroll_len)
    (_, new_learner_state, new_actor_state, new_value_state), train_metrics = (
        jax.lax.scan(
            _inner_step,
            (
                update_rule_params,
                agent_state.learner_state,
                agent_state.actor_state,
                agent_state.value_state,
            ),
            (train_rollouts, learner_rngs),
        )
    )
    train_meta_out = train_metrics.pop('meta_out')

    # Replay saved policy noise and executed model actions, including bootstrap.
    agent_out_on_valid, _ = agent.unroll_net(
        new_learner_state.params,
        valid_rollout.first_state(time_axis=0),
        valid_rollout,
    )

    # Calculate value_fn on the validation rollout.
    value_out, _, _, _ = agent.value_fn.get_value_outs(
        new_value_state, valid_rollout, agent_out_on_valid['logits']
    )
    actions_on_valid = valid_rollout.actions[:-1]
    samples_on_valid = agent_out_on_valid['logits'][:-1]
    adv_t = jax.lax.stop_gradient(value_out.normalized_adv)

    # Calculate meta loss' components.
    # Policy gradient loss.
    pg_loss_per_step = continuous_policies.kde_policy_gradient_loss(
        samples_on_valid, actions_on_valid, advantages=adv_t,
        backprop=False, **kde_kwargs,
    )

    # Meta regularizers.
    reg_loss = 0
    policy_entropy = continuous_policies.kde_entropy(
        samples_on_valid, **kde_kwargs
    ).mean()
    reg_loss += -1e-2 * policy_entropy

    # Validation regularisers.
    z_a = agent_out_on_valid['z_a'][:-1]
    y_entropy_loss = -jnp.mean(
        distrax.Softmax(agent_out_on_valid['y']).entropy()
    )
    z_entropy_loss = -jnp.mean(distrax.Softmax(z_a).entropy())
    reg_loss += 1e-3 * (y_entropy_loss + z_entropy_loss)

    # Train regularisers.
    dp, dy, dz = train_meta_out['pi'], train_meta_out['y'], train_meta_out['z']
    chex.assert_equal_shape_prefix([dp, dy, dz], 3)  # [N, T, B, ...]
    reg_loss += 1e-3 * jnp.mean(jnp.square(jnp.mean(dy, axis=(1, 2, 3))))
    reg_loss += 1e-3 * jnp.mean(jnp.square(jnp.mean(dz, axis=(1, 2, 3))))
    reg_loss += 1e-3 * jnp.mean(jnp.square(jnp.mean(dp, axis=(1, 2, 3))))
    target_samples = train_meta_out['target_out']['logits'][:, :-1]
    chex.assert_equal_shape([target_samples, dp])  # [inner, T, B, samples]
    target_w1 = continuous_policies.smooth_wasserstein_1(
        jax.lax.stop_gradient(target_samples), dp,
        smoothing=wasserstein_smoothing,
    )
    reg_loss += 1e-2 * jnp.mean(target_w1)
    reg_loss *= regularizer_scale

    # Meta loss.
    meta_loss = pg_loss_per_step.mean() + reg_loss

    # Update logs and states.
    meta_log = dict(
        adv=value_out.adv.mean(),
        normalized_adv=value_out.normalized_adv.mean(),
        validation_reward_abs=jnp.abs(valid_rollout.rewards[1:]).sum(),
        entropy=policy_entropy,
        y_entropy=-y_entropy_loss,
        z_entropy=-z_entropy_loss,
        target_w1=target_w1.mean(),
        value=value_out.value.mean(),
        val_importance_weight=jnp.mean(jnp.minimum(value_out.rho, 1.0)),
        meta_loss=meta_loss,
        pg_loss=pg_loss_per_step.mean(),
        reg_loss=reg_loss,
    )
    new_agent_state = MetaTrainState(
        learner_state=new_learner_state,
        actor_state=new_actor_state,
        value_state=new_value_state,
        env_state=agent_state.env_state,
        env_timestep=agent_state.env_timestep,
    )

    return meta_loss, (new_agent_state, train_metrics, meta_log)

  # Calculate meta gradients.
  meta_grads, outputs = jax.grad(_outer_loss, has_aux=True)(
      update_rule_params, agent_state, train_rollouts, valid_rollout, rng
  )
  new_agent_state, train_metrics, meta_log = outputs
  if axis_name is not None:
    (meta_grads, train_metrics, meta_log) = jax.lax.pmean(
        (meta_grads, train_metrics, meta_log), axis_name
    )

  return meta_grads, (new_agent_state, train_metrics, meta_log)

In [ ]:
# @title Meta update step (requires calculating meta gradients for all agents).


def meta_update(
    update_rule_params: types.MetaParams,
    meta_opt_state: optax.OptState,
    agents_states: list[MetaTrainState],
    rng: chex.PRNGKey,
    axis_name: str | None = axis_name,
    regularizer_scale: float = 1.0,
):
  """Calculates an update for the meta parameters."""

  # Generate `num_inner_steps` trajectories for each of `num_agents` agents.
  train_rollouts = [None] * num_agents
  valid_rollouts = [None] * num_agents
  rng_act, rng_upd = jax.random.split(rng)
  rngs_per_agent_act = jax.random.split(rng_act, num_agents)
  for agent_i in range(num_agents):
    agent, state = agents[agent_i], agents_states[agent_i]
    rollouts = [None] * num_inner_steps
    rngs_per_step = jax.random.split(
        rngs_per_agent_act[agent_i], num_inner_steps + 1
    )
    for step_i in range(num_inner_steps):
      state, rollouts[step_i] = agent.unroll_actor(
          state, rngs_per_step[step_i], rollout_len
      )
    train_rollouts[agent_i] = utils.tree_stack(rollouts)

    # Generate 2x longer trajectories for validation.
    agents_states[agent_i], valid_rollouts[agent_i] = agent.unroll_actor(
        state, rngs_per_step[-1], 2 * rollout_len
    )

  # Calculate meta gradients for each agent.
  meta_grads = [None] * num_agents
  rngs_per_agent_upd = jax.random.split(rng_upd, num_agents)
  metrics_per_agent, logs_per_agent = [], []
  for agent_i in range(num_agents):
    meta_grads[agent_i], (agents_states[agent_i], metrics, meta_log) = (
        calculate_meta_gradient(
            update_rule_params=update_rule_params,
            agent_state=agents_states[agent_i],
            train_rollouts=train_rollouts[agent_i],
            valid_rollout=valid_rollouts[agent_i],
            rng=rngs_per_agent_upd[agent_i],
            agent=agents[agent_i],
            axis_name=axis_name,
            regularizer_scale=regularizer_scale,
        )
    )
    metrics_per_agent.append(metrics)
    logs_per_agent.append(meta_log)
  metrics = jax.tree.map(lambda x: x.mean(0), utils.tree_stack(metrics_per_agent))
  meta_log = jax.tree.map(lambda x: x.mean(0), utils.tree_stack(logs_per_agent))

  # Log rewards and proportion positive rewards.
  rewards = [None] * num_agents
  pos_rewards = [None] * num_agents
  neg_rewards = [None] * num_agents
  for agent_i in range(num_agents):
    assert train_rollouts[agent_i] is not None
    r = train_rollouts[agent_i].rewards
    rewards[agent_i] = r.mean()

    pos_rewards[agent_i] = (r > 0).sum()
    neg_rewards[agent_i] = (r < 0).sum()

  # Average meta gradients across all agents and pass through meta optimizer.
  # Note that we had a separate meta optimiser for each agent in the original
  # work; that logic is omitted here for simplicity.
  avg_meta_gradient = jax.tree.map(
      lambda x: x.mean(axis=0), utils.tree_stack(meta_grads)
  )
  meta_update, meta_opt_state = meta_opt.update(
      avg_meta_gradient, meta_opt_state
  )
  update_rule_params = optax.apply_updates(update_rule_params, meta_update)

  meta_log['meta_grad_norm'] = optax.global_norm(avg_meta_gradient)
  meta_log['meta_up_norm'] = optax.global_norm(meta_update)
  meta_log['rewards'] = utils.tree_stack(rewards).mean()
  meta_log['pos_rewards'] = utils.tree_stack(pos_rewards).mean()
  meta_log['neg_rewards'] = utils.tree_stack(neg_rewards).mean()
  return update_rule_params, meta_opt_state, agents_states, metrics, meta_log

In [ ]:
# @title Meta training config

num_steps = 100  # Increase explicitly for an actual training experiment.
num_agents = 2
rollout_len = 32  # Includes the final bootstrap timestep; keep >= 2.
num_inner_steps = 4
batch_size_per_device = 2
rng_key = jax.random.PRNGKey(4)

value_fn_config = types.ValueFnConfig(
    net='mlp',
    net_args=dict(
        dense=(32, 32),
        head_w_init_std=1e-2,
        action_spec=(),
    ),
    learning_rate=1e-2,
    max_abs_update=1.0,
    discount_factor=0.99,
    td_lambda=0.96,
    outer_value_cost=1.0,
    continuous_actions=True,
    kde_bandwidth=kde_bandwidth,
    action_minimum=action_minimum,
    action_maximum=action_maximum,
)

# Use random params for the update rule.
update_rule_params = random_update_rule_params
meta_opt = optax.adam(learning_rate=5e-4)

meta_opt_state = meta_opt.init(update_rule_params)
meta_log = {}
metrics = {}

# Create multiple agents.
agents = []
agents_states = []
rng, rng_key = jax.random.split(rng_key)
for rng_key in jax.random.split(rng_key, num_agents):
  agents.append(
      MetaTrainAgent(
          batch_size_per_device=batch_size_per_device,
          agent_settings=agent_settings,
          value_fn_config=value_fn_config,
      )
  )
  agents_states.append(agents[-1].init_state(rng_key))

# Parallelise using all available devices.
devices = jax.devices()
jitted_meta_update = jax.pmap(meta_update, axis_name=axis_name, devices=devices)

In [ ]:
# @title Run meta-training (note that compilation can take time!).

# Replicate parameters for each device.
step_update_rule_params = jax.device_put_replicated(
    update_rule_params, devices
)
step_meta_opt_state = jax.device_put_replicated(meta_opt_state, devices)
step_agents_states = jax.device_put_replicated(agents_states, devices)

# Run the meta learning loop.
for meta_step in tqdm.tqdm(range(num_steps)):
  if meta_step in metrics:  # to support interrupting and continuing
    continue

  rng, step_rngs = jax.random.split(rng)
  step_rngs = jax.random.split(step_rngs, len(devices))
  (
      step_update_rule_params,
      step_meta_opt_state,
      step_agents_states,
      metrics[meta_step],
      meta_log[meta_step],
  ) = jitted_meta_update(
      update_rule_params=step_update_rule_params,
      meta_opt_state=step_meta_opt_state,
      agents_states=step_agents_states,
      rng=step_rngs,
  )
  metrics[meta_step], meta_log[meta_step] = jax.device_get(
      (metrics[meta_step], meta_log[meta_step])
  )

# Collect metrics from all devices.
metrics, meta_log = utils.gather_from_devices((metrics, meta_log))
metrics, meta_log = jax.tree.map(lambda x: x.mean(0), (metrics, meta_log))

In [ ]:
# @title Process logs and metrics.

meta_log_cpu = jax.device_get(meta_log)
steps = np.sort(np.unique(list(meta_log_cpu.keys())))
rows = []
for i in steps:
  for key in (
      'meta_grad_norm',
      'meta_up_norm',
      'meta_loss',
      'pg_loss',
      'entropy',
      'target_w1',
      'rewards',
      'pos_rewards',
      'neg_rewards',
  ):
    rows.append(dict(step=i, value=float(meta_log_cpu[i][key]), f=key))

df = pd.DataFrame(rows)

In [ ]:
sns.relplot(
    data=df[df.f.isin(['pos_rewards', 'neg_rewards'])],
    x='step',
    y='value',
    kind='line',
    hue='f',
    errorbar=None,
    aspect=1.5,
)
plt.show()

In [ ]:
sns.relplot(
    data=df[
        df.f.isin(['meta_grad_norm', 'meta_loss', 'meta_up_norm', 'rewards',
                   'entropy', 'target_w1'])
    ],
    x='step',
    y='value',
    kind='line',
    col='f',
    errorbar=None,
    aspect=1.5,
    facet_kws={'sharey': False, 'sharex': True},
)
plt.show()

## Experiment 1: Confirm continuous catch is solvable

In [ ]:
# @title PPO baseline: can the unchanged continuous Catch environment be learned?
# Run setup through get_env first. PPO is independent of Disco.
from disco_rl.ppo import PPOConfig, train_ppo, save_training_results

ppo_config = PPOConfig(
    seeds=(301, 302, 303),
    num_updates=100,
    num_envs=32,
    horizon=128,
    checkpoint_steps=(12_288, 24_576),
)
ppo_output_dir = Path(repo_root) / 'results' / 'continuous_catch_ppo'
ppo_result = train_ppo(get_env, ppo_config)
ppo_training_results = ppo_result.training
ppo_evaluation_results = ppo_result.evaluation
ppo_final_params = ppo_result.final_params
ppo_fig = save_training_results(ppo_result, ppo_output_dir)
plt.show()


In [ ]:
# @title One Catch episode: initial, early, and fully trained PPO.
from IPython.display import HTML, display
from disco_rl.ppo import visualize_episode

ppo_viz_training_seed = 302
ppo_viz_episode_seed = 0
ppo_viz_mid_steps = 12_288

ppo_comparison = visualize_episode(
    ppo_result,
    training_seed=ppo_viz_training_seed,
    episode_seed=ppo_viz_episode_seed,
    middle_steps=ppo_viz_mid_steps,
    output_dir=ppo_output_dir,
)
ppo_episode_comparison = ppo_comparison.summary
display(HTML(ppo_comparison.html))


## Experiment 2: Checking smooth Wasserstein works

In [ ]:
# Run setup cells through the actor-unroll helper first; meta-training is optional.
import rlax
from disco_rl.value_fns import value_utils

fit_rule_source = 'random'  # 'trained' selects device 0 of step_update_rule_params.
fit_num_updates = 200
fit_batch_size = 8
fit_rollout_len = 64
fit_seed = 101

if fit_rule_source == 'random':
  fit_rule_params = random_update_rule_params
elif fit_rule_source == 'trained':
  if 'step_update_rule_params' not in globals():
    raise RuntimeError('Run meta-training before selecting a trained rule.')
  fit_rule_params = jax.tree.map(lambda x: x[0], step_update_rule_params)
else:
  raise ValueError("fit_rule_source must be 'random' or 'trained'.")
fit_rule_params = jax.tree.map(jax.lax.stop_gradient, fit_rule_params)
if fit_num_updates < 1 or fit_batch_size < 1 or fit_rollout_len < 2:
  raise ValueError('Use positive updates/batch size and rollout_len >= 2.')

fit_env = get_env(fit_batch_size)
fit_agent = agent_lib.Agent(
    single_observation_spec=fit_env.single_observation_spec(),
    single_action_spec=fit_env.single_action_spec(),
    agent_settings=agent_settings, batch_axis_name=None,
)
fit_keys = jax.random.split(jax.random.PRNGKey(fit_seed), 5)
fit_learner = fit_agent.initial_learner_state(fit_keys[0])
fit_actor_start = fit_agent.initial_actor_state(fit_keys[1])
fit_env_state, fit_timestep = fit_env.reset(fit_keys[2])
fit_collect = jax.jit(partial(
    unroll_jittable_actor, env=fit_env, rollout_len=fit_rollout_len,
    actor_step_fn=fit_agent.actor_step,
))
fit_rollout, _, _, _ = fit_collect(
    fit_learner.params, fit_actor_start, fit_timestep,
    fit_env_state, fit_keys[3],
)
fit_initial_out, _ = fit_agent.unroll_net(
    fit_learner.params, fit_actor_start, fit_rollout
)
fit_inputs = types.UpdateRuleInputs(
    observations=fit_rollout.observations, actions=fit_rollout.actions,
    rewards=fit_rollout.rewards[1:],
    is_terminal=fit_rollout.discounts[1:] == 0,
    behaviour_agent_out=fit_rollout.agent_outs, agent_out=fit_initial_out,
)
fit_hyper = fit_agent.settings.hyper_params.to_dict()
fit_meta_out, _ = fit_agent.update_rule.unroll_meta_net(
    meta_params=fit_rule_params, params=fit_learner.params,
    state=fit_actor_start, meta_state=fit_learner.meta_state,
    rollout=fit_inputs, hyper_params=fit_hyper,
    unroll_policy_fn=fit_agent._network.unroll, rng=fit_keys[4], axis_name=None,
)
# Freeze absolute value targets, not TD errors (which would move with Q).
fit_value_target = value_utils.get_values_from_net_outs(
    fit_initial_out['q_a'][:-1], categorical_value=True,
    nonlinear_transform=True,
    max_abs_value=fit_agent.settings.update_rule.max_abs_value,
) + fit_meta_out['q_td']
fit_targets = jax.tree.map(jax.lax.stop_gradient, dict(
    pi=fit_meta_out['pi'], y=fit_meta_out['y'], z=fit_meta_out['z'],
    aux_pi=fit_initial_out['logits'][1:], value=fit_value_target,
))
fit_masks = (fit_rollout.discounts[:-1] > 0).astype(jnp.float32)
if float(fit_masks.sum()) == 0:
  raise RuntimeError('The fixed rollout contains no valid training transitions.')

def fit_masked_mean(values):
  return (values * fit_masks).sum() / fit_masks.sum()

def fit_loss(params):
  # The rollout includes stored policy noise; the starting RNN state never moves.
  out, _ = fit_agent.unroll_net(params, fit_actor_start, fit_rollout)
  losses = dict(
      pi_loss=continuous_policies.smooth_wasserstein_1(
          fit_targets['pi'], out['logits'][:-1], wasserstein_smoothing),
      y_loss=rlax.categorical_kl_divergence(fit_targets['y'], out['y'][:-1]),
      z_loss=rlax.categorical_kl_divergence(fit_targets['z'], out['z_a'][:-1]),
      aux_loss=continuous_policies.smooth_wasserstein_1(
          fit_targets['aux_pi'], out['aux_pi_a'][:-1], wasserstein_smoothing
      ) * (1.0 - fit_inputs.is_terminal),
      q_loss=value_utils.value_loss_from_target(
          out['q_a'][:-1], fit_targets['value'], nonlinear_transform=True,
          categorical_value=True,
          max_abs_value=fit_agent.settings.update_rule.max_abs_value),
  )
  losses = jax.tree.map(fit_masked_mean, losses)
  total = sum(fit_hyper[cost] * losses[name] for cost, name in (
      ('pi_cost', 'pi_loss'), ('y_cost', 'y_loss'), ('z_cost', 'z_loss'),
      ('aux_policy_cost', 'aux_loss'), ('value_cost', 'q_loss'),
  ))
  samples = out['logits'][:-1]
  span = action_maximum - action_minimum
  near_bound = ((samples <= action_minimum + 0.01 * span)
                | (samples >= action_maximum - 0.01 * span))
  return total, dict(
      total_loss=total, **losses,
      policy_spread=fit_masked_mean(jnp.std(samples, axis=-1)),
      bound_fraction=fit_masked_mean(near_bound.mean(axis=-1)),
  )

fit_measure = jax.jit(jax.value_and_grad(fit_loss, has_aux=True))

@jax.jit
def fit_apply(params, opt_state, grads):
  updates, opt_state = fit_agent._optimizer.update(grads, opt_state, params)
  return optax.apply_updates(params, updates), opt_state, optax.global_norm(updates)

fit_params, fit_opt_state = fit_learner.params, fit_learner.opt_state
fit_rows = []
fit_update_norm = jnp.array(0.0)
for fit_step in tqdm.tqdm(range(fit_num_updates + 1), desc='Fixed targets'):
  (_, fit_log), fit_grads = fit_measure(fit_params)
  fit_log = dict(fit_log, gradient_norm=optax.global_norm(fit_grads),
                 update_norm=fit_update_norm)
  fit_row = {key: float(value) for key, value in jax.device_get(fit_log).items()}
  if not np.isfinite(list(fit_row.values())).all():
    raise FloatingPointError(f'Non-finite fixed-target metrics at update {fit_step}.')
  fit_rows.append(dict(update=fit_step, **fit_row))
  if fit_step < fit_num_updates:
    fit_params, fit_opt_state, fit_update_norm = fit_apply(
        fit_params, fit_opt_state, fit_grads)

fixed_target_results = pd.DataFrame(fit_rows)
print(f'Frozen rule: {fit_rule_source}. This tests fitting, not reward improvement.')
print(fixed_target_results.iloc[[0, -1]].to_string(index=False))
if fixed_target_results.total_loss.iloc[-1] >= fixed_target_results.total_loss.iloc[0]:
  print('WARNING: fixed-target loss did not decrease; inspect components and gradients.')
fit_fig, fit_axes = plt.subplots(4, 2, figsize=(16, 16))
fit_axes = fit_axes.ravel()

metrics = ['total_loss', 'pi_loss', 'y_loss', 'z_loss', 'aux_loss', 'q_loss']
for i in range(6):
  fixed_target_results.plot(x='update', y=metrics[i], ax=fit_axes[i])
  # fit_axes[i].set_yscale('symlog', linthresh=1e-5)
  fit_axes[i].set_title('Fixed data AND fixed targets')
fixed_target_results.plot(x='update', y=['gradient_norm', 'update_norm'], ax=fit_axes[6])
fit_axes[6].set_yscale('symlog', linthresh=1e-8)
fixed_target_results.plot(x='update', y=['policy_spread', 'bound_fraction'], ax=fit_axes[7])
fit_fig.tight_layout()
plt.show()
fit_fig.savefig("../results/wasserstein_results.png")


## Experiment 3: KDE Validation

#### Experiment 3(a): KDE Numerical correctness

Check bounded-density normalisation, boundary support, and agreement with an independent truncated-Gaussian reference. Compare first and second derivatives with finite differences, including the detached action/advantage paths used for policy gradients. Passing validates the implemented KDE formula, not its accuracy as an actor likelihood.

In [ ]:
from pathlib import Path
from IPython.display import display
from matplotlib import pyplot as plt
from disco_rl.kde_validation import validate_numerics, save_validation

kde_numerical_result = validate_numerics(bandwidths=(0.03, 0.1, 0.3))
display(kde_numerical_result.tables['checks'].head(), kde_numerical_result.tables['checks'].tail())
save_validation(kde_numerical_result, Path(repo_root) / 'results' / 'kde_validation' / 'numerical')
plt.show()
if not kde_numerical_result.tables['checks']['passed'].all():
  raise AssertionError('KDE numerical checks failed; inspect the reported errors and tolerances.')


#### Experiment 3(b): KDE Statistical accuracy

Compare KDE with exact tanh-Gaussian densities for central, narrow, and near-boundary policies. Sweep sample count and bandwidth across repeated seeds. Measure density error, self-bank versus held-out entropy, and importance-ratio error. These are bias/variance measurements, not universal pass/fail criteria.

In [ ]:
from pathlib import Path
from IPython.display import display
from matplotlib import pyplot as plt
from disco_rl.kde_validation import validate_accuracy, save_validation

kde_accuracy_result = validate_accuracy(
    sample_sizes=(32, 128, 512), bandwidths=(0.03, 0.1, 0.3),
    seeds=(601, 602, 603),
)
kde_accuracy_table = kde_accuracy_result.tables['accuracy']
display(kde_accuracy_table.groupby(['case', 'samples', 'bandwidth'])[[
    'density_l1', 'boundary_l1', 'log_density_rmse',
    'self_entropy_error', 'heldout_entropy_error',
    'ratio_log_rmse', 'self_ratio_log_rmse', 'clipped_ratio_mae',
]].mean())
save_validation(kde_accuracy_result, Path(repo_root) / 'results' / 'kde_validation' / 'accuracy')
plt.show()


#### Experiment 3(c): KDE Learning-gradient usefulness

Use a one-step bounded-action task with reward $r(a)=-(a-0.6)^2$ and a tanh-Gaussian policy with learnable mean and log standard deviation. Compare exact score gradients and KDE gradients against an independently computed expected-return gradient. The self-bank case includes the executed action in its density bank, as the Disco actor does; an independent-bank case isolates that sampling dependence. Then train all three estimators and plot actual expected reward. This isolates density-based learning; it is not a Catch or meta-learning experiment.

In [ ]:
from pathlib import Path
from IPython.display import display
from matplotlib import pyplot as plt
from disco_rl.kde_validation import validate_learning, save_validation

kde_learning_result = validate_learning(
    sample_sizes=(32, 128, 512), bandwidths=(0.03, 0.1, 0.3),
    seeds=(701, 702, 703), action_batches=256,
    learning_samples=128, learning_bandwidth=0.1,
    num_updates=100, learning_rate=0.1,
)
display(kde_learning_result.tables['gradients'].groupby([
    'estimator', 'samples', 'bandwidth',
])[['cosine', 'relative_gradient_error', 'gradient_norm_ratio', 'measured_return_change']].mean())
kde_learning_table = kde_learning_result.tables['learning']
display(kde_learning_table[kde_learning_table['update'].isin((0, kde_learning_table['update'].max()))].groupby([
    'estimator', 'update',
])['expected_return'].agg(['mean', 'std']))
save_validation(kde_learning_result, Path(repo_root) / 'results' / 'kde_validation' / 'learning')
plt.show()


## Experiment 4: Direct PPO policy imitation

In [ ]:
# @title Configure policy-only imitation of a frozen PPO teacher.
from disco_rl.ppo.imitation import (
    ImitationConfig, train_imitation, save_imitation_results,
)

if 'ppo_result' not in globals():
  raise RuntimeError('Run the PPO baseline cell first to provide the trained teacher.')

imitation_config = ImitationConfig(
    teacher_seed=302,
    seeds=(401, 402, 403),
    num_updates=500,
    num_envs=8,
    rollout_len=32,
    updates_per_rollout=1,
    eval_every=50,
)
imitation_output_dir = Path(repo_root) / 'results' / 'continuous_catch_ppo_imitation'
print('Frozen PPO distribution targets; only pi loss. No y/z/p/q or meta-updates.')


In [ ]:
# @title Train fresh Disco agents toward PPO using only smooth Wasserstein.
imitation_result = train_imitation(ppo_result, agent_settings, imitation_config)
imitation_training_results = imitation_result.training
imitation_evaluation_results = imitation_result.evaluation


In [ ]:
# @title Plot policy matching and catch rate, and save the reward histories.
imitation_fig = save_imitation_results(imitation_result, imitation_output_dir)
plt.show()
# Sampled student actions are the main comparison with stochastic PPO.
# Mean-action evaluation diagnoses whether residual exploration hurts precision.


## Experiment 5(a): PPO guided Meta Learning

Adam monitoring: rerun the configuration, training, and plot/save cells below to collect new optimiser history. Old loss curves cannot reconstruct Adam's buffers. After every outer update, record summaries of the actual first moment `u`, `sqrt(v)`, bias-corrected `u_hat` and `sqrt(v_hat)`, and the effective ratio `-eta_update / outer_learning_rate`. Moments incorporate the globally clipped mean meta-gradient, not the unclipped gradient. Both rates and the learning rule are unchanged.

The plot/save cell now displays an extra Adam figure. Signed coordinate medians and 5th-95th percentile bands avoid cancellation in a single mean; RMS curves compare raw and bias-corrected scales. `adam_diagnostics.csv` retains these statistics, min/max and coordinate counts for each parameter tensor and all coordinates combined (`__all__`), not a full copy of every coordinate at every update. `training.csv` also includes the global RMS traces, and progress logs print raw moment and ratio RMS values. Full summaries accumulate in memory during training and are exported by the plot/save cell.

A shrinking `u` relative to the denominator can suppress steps; persistent momentum can also lag changing gradients. Raw moments have startup bias, so inspect corrected moments and the actual ratio too. These plots describe scaling; they do not prove the meta-gradient points towards a better PPO-like policy. The plotted outer loss is measured before the eta update, on changing students/data, not a paired before/after test of that update. Results are saved alongside the usual experiment outputs as `adam_diagnostics.csv` and `adam_diagnostics.png`.


In [ ]:
# @title Configure meta-learning with a frozen PPO outer objective.
from importlib import reload
from disco_rl.ppo import meta_training as ppo_meta_module
reload(ppo_meta_module)
from disco_rl.ppo.meta_training import (
    MetaTrainingConfig, train_meta, save_meta_results,
)

if 'ppo_result' not in globals():
  raise RuntimeError('Run the PPO baseline first to provide the frozen teacher.')

ppo_meta_config = MetaTrainingConfig(
    teacher_seed=302,
    meta_seed=501,
    seeds=(501, 502, 511, 512),
    num_updates=5000,
    num_inner_steps=8,
    num_envs=4,
    rollout_len=16,
    meta_learning_rate=3e-4,
    eval_every=50,
    fresh_seeds=(521, 522, 523),
    fresh_updates=500,
)
ppo_meta_output_dir = Path(repo_root) / 'results' / 'continuous_catch_ppo_meta'
print('Inner: Disco policy targets only. Outer: smooth Wasserstein to frozen PPO.')
print('No A2C, entropy regularizers, or y/z/p/q losses; rewards are evaluation only.')
print(f'{ppo_meta_config.transitions_per_update:,} collected transitions per meta-update, including validation.')


In [ ]:
# @title Train a new meta-network through differentiable agent updates.
# Independent of the original meta-training and direct-imitation states.
ppo_meta_result = train_meta(ppo_result, agent_settings, ppo_meta_config)
ppo_meta_training_results = ppo_meta_result.training
ppo_meta_evaluation_results = ppo_meta_result.evaluation
ppo_meta_fresh_results = ppo_meta_result.fresh_evaluation


In [ ]:
# @title Plot/save outer loss and catch rates, including fresh-agent rule evaluation.
from importlib import reload
from disco_rl.ppo import meta_training as ppo_meta_module
reload(ppo_meta_module)
save_meta_results = ppo_meta_module.save_meta_results
ppo_meta_fig = save_meta_results(ppo_meta_result, ppo_meta_output_dir)
plt.show()
display(ppo_meta_training_results.tail())
display(ppo_meta_fresh_results[
    ppo_meta_fresh_results['update'].isin((0, ppo_meta_config.fresh_updates))
])
# Fresh agents learn only from the frozen discovered rule, never PPO targets.
# Policy distance/catch rate need not improve: this tests rule discovery.


## Experiment 5(b): PPO-guided meta-learning with full meta network targets

In [ ]:
# @title Repeat the policy-only experiment with the full Disco inner loss.
from dataclasses import asdict
import importlib
from disco_rl.update_rules import disco as disco_rule_module
from disco_rl.ppo import meta_training as ppo_meta_module
# Load the new experiment without discarding the completed baseline in this kernel.
importlib.reload(disco_rule_module)
importlib.reload(ppo_meta_module)
from disco_rl.ppo.meta_training import MetaTrainingConfig, train_meta, save_meta_results

if 'ppo_meta_result' not in globals() or 'ppo_result' not in globals():
  raise RuntimeError('Run the policy-only PPO meta-training experiment first for a matched baseline.')
if getattr(ppo_meta_result.trainer.config, 'use_auxiliary_losses', False):
  raise ValueError('The comparison baseline must use policy-only inner learning.')

ppo_full_meta_config = MetaTrainingConfig(**{
    **asdict(ppo_meta_result.trainer.config), 'use_auxiliary_losses': True,
})
ppo_full_agent_settings = config_dict.ConfigDict(
    ppo_meta_result.trainer.agent.settings.to_dict()
)
for cost in ('y_cost', 'z_cost', 'aux_policy_cost', 'value_cost'):
  ppo_full_agent_settings.hyper_params[cost] = agent_settings.hyper_params[cost]
baseline_teacher = ppo_meta_result.trainer.student.teacher_params
current_teacher = ppo_result.final_params[ppo_full_meta_config.teacher_seed]
if (jax.tree.structure(baseline_teacher) != jax.tree.structure(current_teacher)
    or not all(np.array_equal(a, b) for a, b in zip(
        jax.tree.leaves(baseline_teacher), jax.tree.leaves(current_teacher)))):
  raise ValueError('PPO teacher changed since the baseline; rerun the policy-only experiment.')
ppo_full_meta_output_dir = Path(repo_root) / 'results' / 'continuous_catch_ppo_meta_full'
print('Same seeds, architecture, sample count, budgets, and frozen PPO outer loss as the baseline.')
print('Only change: original y/z/p/q inner losses are enabled, including inner value learning.')
print('Sample bank:', ppo_full_agent_settings.update_rule.num_samples)
print(dict(ppo_full_agent_settings.hyper_params))


In [ ]:
# @title Train a fresh meta-network with full inner learning, then test its rule on fresh agents.
ppo_full_meta_result = train_meta(
    ppo_result, ppo_full_agent_settings, ppo_full_meta_config,
)
ppo_full_meta_training_results = ppo_full_meta_result.training
ppo_full_meta_evaluation_results = ppo_full_meta_result.evaluation
ppo_full_meta_fresh_results = ppo_full_meta_result.fresh_evaluation


In [ ]:
# @title Save full-loss results and compare held-out performance with policy-only learning.
ppo_full_meta_fig = save_meta_results(ppo_full_meta_result, ppo_full_meta_output_dir)
plt.show()
display(ppo_full_meta_training_results.tail())
ppo_inner_loss_comparison = []
for label, result in (('policy_only', ppo_meta_result), ('full_inner', ppo_full_meta_result)):
  for stage, frame in (('meta_train', result.evaluation), ('fresh_rule', result.fresh_evaluation)):
    endpoints = frame[frame['update'].isin((0, frame['update'].max()))]
    summary = endpoints.groupby(['update', 'mode'])[['pi_loss', 'catch_rate', 'mean_return']].mean().reset_index()
    summary.insert(0, 'experiment', label)
    summary.insert(1, 'stage', stage)
    ppo_inner_loss_comparison.append(summary)
ppo_inner_loss_comparison = pd.concat(ppo_inner_loss_comparison, ignore_index=True)
display(ppo_inner_loss_comparison)
ppo_inner_loss_comparison.to_csv(ppo_full_meta_output_dir / 'comparison.csv', index=False)
# Inner component losses have different meanings/scales; outer pi_loss and catch rate are comparable.


In [ ]:
display(ppo_meta_training_results.tail())
display(ppo_full_meta_training_results.tail())